# FETCH FILE

In [11]:
import pandas as pd
import numpy as np
from google.colab import files

df = pd.read_sas("/content/DR1IFF_J.xpt", format="xport", encoding="utf-8")
print(df.shape)

(112683, 84)


# EXTRACTION

In [12]:
# Filter to red meat food codes
# Beef: 21xxxxxx, Pork: 23xxxxxx, Lamb/Veal: 22xxxxxx
red_meat_mask = (
    ((df["DR1IFDCD"] >= 21000000) & (df["DR1IFDCD"] <= 21999999)) |
    ((df["DR1IFDCD"] >= 22000000) & (df["DR1IFDCD"] <= 22999999)) |
    ((df["DR1IFDCD"] >= 23000000) & (df["DR1IFDCD"] <= 23999999))
)

df_red_meat = df[red_meat_mask]

# Sum grams per person per day
red_meat_per_person = (
    df_red_meat
    .groupby("SEQN")["DR1IGRMS"]
    .sum()
    .reset_index()
)
red_meat_per_person.columns = ["SEQN", "red_meat_g"]

# Include non-consumers (zero intake days)
all_participants = df["SEQN"].unique()
all_df = pd.DataFrame({"SEQN": all_participants})
red_meat_final = all_df.merge(red_meat_per_person, on="SEQN", how="left")
red_meat_final["red_meat_g"] = red_meat_final["red_meat_g"].fillna(0)

# Compute stats
print(f"N participants:  {len(red_meat_final)}")
print(f"Mean:   {red_meat_final['red_meat_g'].mean():.2f} g/day")
print(f"SD:     {red_meat_final['red_meat_g'].std():.2f} g/day")
print(f"Median: {red_meat_final['red_meat_g'].median():.2f} g/day")
print(f"% zero intake: {(red_meat_final['red_meat_g']==0).mean()*100:.1f}%")

N participants:  7640
Mean:   20.13 g/day
SD:     58.12 g/day
Median: 0.00 g/day
% zero intake: 79.1%


In [13]:
# DR1IFDCD food category variable — check what categories exist
# Red meat categories in WWEIA are typically:
# Category 2601 = Beef, not further specified
# Category 2602 = Ground beef
# Category 2603 = Beef, lean only eaten
# etc.

print(df["DR1IWSZ"].value_counts())  # or whichever category column exists
print([col for col in df.columns if 'CAT' in col.upper()])

KeyError: 'DR1IWSZ'